# Advanced Data Analysis and Preprocessing

## Text Message Sentiment Analysis

This notebook documents the following analyses:

1. Chi-square vocabulary analysis
2. Conflicting-label detection
3. Text artifact detection and cleaning
4. Duplicate detection and handling
5. Construction of two dataset variants
6. Leakage-free train, validation, and test splits

The preprocessing decisions and results will be documented as each analysis is implemented.


In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset

# Load the same dataset used in the main project
dataset = load_dataset("dair-ai/emotion", "unsplit")

df = dataset["train"].to_pandas()

print("Dataset shape:", df.shape)
print(df["label"].value_counts())
df.head()


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

unsplit/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 26.9MB            

unsplit/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/416809 [00:00<?, ? examples/s]

Dataset shape: (416809, 2)
label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64


,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
4,i beleive that i am much more sensitive to oth...,2


In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

#Create a word-count representation
vectorizer = CountVectorizer(
    stop_words="english",
    min_df=5,
    max_features=20000
)

X = vectorizer.fit_transform(df["text"].fillna(""))
words = vectorizer.get_feature_names_out()

#calculate chi-square scores for each emotion
emotion_names = [
    "sadness", "joy", "love",
    "anger", "fear", "surprise"
]

chi2_results = {}

for emotion_id, emotion in enumerate(emotion_names):
    y_binary = (df["label"] == emotion_id).astype(int)

    scores, p_values = chi2(X, y_binary)

    results = pd.DataFrame({
        "word": words,
        "chi2_score": scores,
        "p_value": p_values
    })

    results = results.sort_values(
        "chi2_score", ascending=False
    )

    chi2_results[emotion] = results

    print(f"\nTop 15 words associated with {emotion}:")
    display(results.head(15))



Top 15 words associated with sadness:


,word,chi2_score,p_value
19772,worthless,3312.956323,0.0
8322,homesick,3260.703376,0.0
15784,shitty,3250.245086,0.0
9335,isolated,3230.567681,0.0
152,aching,3196.456719,0.0
5415,dull,3192.342410,0.0
10068,lethargic,3164.127073,0.0
18895,useless,3155.300055,0.0
7714,groggy,3150.302392,0.0
5203,doomed,3139.375017,0.0



Top 15 words associated with joy:


,word,chi2_score,p_value
14438,relaxed,2821.862963,0.0
3524,confident,2790.551333,0.0
18953,valued,2657.421616,0.0
19774,worthwhile,2631.393549,0.0
5806,energetic,2625.325359,0.0
3280,comfortable,2608.479440,0.0
18951,valuable,2599.874206,0.0
13544,privileged,2592.721811,0.0
79,acceptable,2585.042352,0.0
17137,successful,2574.486921,0.0



Top 15 words associated with love:


,word,chi2_score,p_value
17423,sympathetic,15122.237910,0.0
17673,tender,14410.597938,0.0
10330,longing,14197.966775,0.0
7337,gentle,14154.573395,0.0
2511,caring,13466.755386,0.0
8387,horny,12367.743228,0.0
11589,naughty,12289.454126,0.0
14981,romantic,12120.583203,0.0
10408,loving,11893.469319,0.0
8417,hot,11505.463766,0.0


... (2 outputs left out)


Top 15 words associated with fear:


,word,chi2_score,p_value
13461,pressured,10558.672551,0.0
800,apprehensive,10260.094395,0.0
14473,reluctant,10257.829286,0.0
8178,hesitant,10052.012279,0.0
9198,intimidated,9998.765971,0.0
15687,shaky,9929.317827,0.0
18808,unsure,9889.341714,0.0
9023,insecure,9645.755555,0.0
12532,paranoid,9575.670212,0.0
18551,uncertain,9526.755955,0.0



Top 15 words associated with surprise:


,word,chi2_score,p_value
8744,impressed,35057.336506,0.000000e+00
578,amazed,34495.974766,0.000000e+00
15791,shocked,32784.369732,0.000000e+00
4170,curious,31112.689418,0.000000e+00
17290,surprised,28910.585801,0.000000e+00
7191,funny,24699.509619,0.000000e+00
4330,dazed,23593.655779,0.000000e+00
17062,stunned,16134.899094,0.000000e+00
16944,strange,14776.394876,0.000000e+00
19454,weird,13908.527518,0.000000e+00
